# Баға және көлем аномалиялары агенті

Ноутбук деректерді PostgreSQL-ден жүктейді, әділ бағаны және көлемнің өсуін есептейді, содан кейін сол нәтижелермен жұмыс істейтін агентті құрады. Ұяшықтарды **жоғарыдан төмен ретімен** іске қосыңыз.

**Әділ баға** (шарт жолдары бойынша):

| Санат | Әділ баға |
|---|---|
| Однородные товары | басқа шарттардағы медиана мен орташа (бұрынғы әдіс) |
| Неоднородные товары | басқа ұйымдардың p25–p75 диапазоны |
| Редкие товары (`small`) | басқа ұйымдардың p25–p75 диапазоны |
| Услуги | басқа ұйымдардың p25–p75 диапазоны |
| Работы | басқа ұйымдардың p25–p75 диапазоны |

**Көлем** (жылдық жоспарлар бойынша, тауарлар, қызметтер мен жұмыстар үшін бірдей): позицияның саны ұйымның **өз** алдыңғы жылдарымен салыстырылады, ал «әдеттегі өзгеріс» сол ұйымның басқа позицияларының сол жылғы өзгерісінен (p25–p75) алынады. Аномалия тек санмен анықталады, ал артық сома тек сұрыптау үшін қолданылады.

Агент орыс тілінде жауап береді және төрт құралды пайдаланады: `fair_price`, `find_anomalies`, `volume_check`, `find_volume_anomalies`.

## 1. Баптаулар

`INFL_PATH` — ИПЦ файлының жолы, `BASE` — барлық баға келтірілетін ай. Көлем шектері (`VOL_...`) уақытша: 7.3-ұяшықтың нәтижесіне қарап бекітеміз. Шектерді өзгертсеңіз, 5-бөлімнен бастап қайта іске қосыңыз.

In [1]:
import os, json
import numpy as np
import pandas as pd
from dotenv import load_dotenv, find_dotenv
from sqlalchemy import create_engine
from openai import OpenAI

load_dotenv(find_dotenv(usecwd=True))
pd.set_option("display.width", 200)

INFL_PATH = "../src/ИПЦ и его составляющие.xlsx"   # ИПЦ файлы
BASE = pd.Timestamp("2026-10-01")                  # бағалар осы айға келтіріледі

# Біртекті/әртекті топтарды анықтау (тауарлар)
TIER_MIN_N = 3                # шашыраңқылықты өлшеуге ең аз жол саны
MAD_LIMIT = np.log(5) / 2     # әдеттегі ауытқу ≈ 2.24 есе
FAR_LIMIT = 0.20              # медианадан 3 есе алшақ жолдардың үлесі
FAR_MIN_N = 5                 # үлесті есептеуге ең аз жол саны

# Диапазон әдісі
MIN_N = 5                     # эталонда осыдан аз жол болса, «Мало строк»
MIN_CUST = 2                  # басқа ұйымдар осыдан аз болса, өз ұйымының жолдары да алынады
MIN_REF_STATUS = 3            # эталонда осыдан аз жол болса, «недостаточно данных»
BY_CUSTOMER = True            # әр ұйымның салмағы бірдей
Q = [0.25, 0.50, 0.75]

# Біртекті тауарлар: data_enough шарты
HOMO_MIN_N, HOMO_MIN_CUST = 5, 2

# Аномалиялар
OVER = 0.30                   # ТЗ шегі
HIGH_X = 3.0                  # «сильно выше»: әділ бағадан 3 есе және одан жоғары
MAX_LIMIT = 30                # бір санат бойынша модельге берілетін ең көп жол

# Көлем аномалиялары (уақытша шектер)
VOL_MIN_GROWTH = 1.5          # сан алдыңғы жылдардың орташасынан кемінде 1.5 есе өсті
VOL_ORG_X = 1.3               # және ұйымның әдеттегі өзгерісінің p75-інен 30%-ға жоғары
VOL_HIGH = 3.0                # «сильно выше»: кемінде 3 есе өсу
VOL_MIN_ORG_ITEMS = 5         # ұйымда салыстыратын позиция осыдан аз болса, 2-шарт қолданылмайды

RU_MONTHS = ["января", "февраля", "марта", "апреля", "мая", "июня", "июля",
             "августа", "сентября", "октября", "ноября", "декабря"]
PRICE_BASIS = f"цены {RU_MONTHS[BASE.month - 1]} {BASE.year} г. с учётом инфляции"

## 2. Деректерді жүктеу

Шамамен 78 мың жол шығуы керек: `subject_type` бойынша 1 — тауарлар, 3 — қызметтер, 2 — жұмыстар.

In [2]:
PG_URL = (
    f"postgresql+psycopg2://{os.environ['POSTGRES_USER']}:{os.environ['POSTGRES_PASSWORD']}"
    f"@{os.getenv('POSTGRES_HOST', 'localhost')}:{os.getenv('POSTGRES_PORT', '5432')}"
    f"/{os.environ['POSTGRES_DB']}"
)
pg = create_engine(PG_URL)

df = pd.read_sql("""
    SELECT cl.line_id, cl.contract_id, cl.lot_id, cl.plan_id,
           cl.customer_bin, cl.supplier_bin,
           cl.year, cl.sign_date, c.plan_exec_date, c.fact_exec_date, cl.status,
           cl.fact_trade_method, cl.is_single_source,
           cl.enstru_code, cl.unit_code, cl.item_name_ru,
           cl.quantity, cl.unit_price, cl.total_sum,
           cl.region_code, cl.region_source,
           p.subject_type,
           p.name_ru         AS plan_name,
           p.desc_ru         AS plan_desc,
           p.extra_desc_ru   AS plan_extra_desc,
           l.name_ru         AS lot_name,
           l.description_ru  AS lot_desc
    FROM contract_lines cl
    JOIN contracts c       ON c.contract_id = cl.contract_id
    LEFT JOIN plan_items p ON p.plan_id = cl.plan_id
    LEFT JOIN lots l       ON l.lot_id  = cl.lot_id
    WHERE cl.price_valid = 1
      AND cl.enstru_code IS NOT NULL AND cl.enstru_code <> ''
""", pg)

df["subject_type"] = df["subject_type"].astype(str).str.replace(r"\.0$", "", regex=True)
df["unit_code"] = df["unit_code"].astype(str)
df["sign_date"] = pd.to_datetime(df["sign_date"], errors="coerce")

print("Жолдар:", len(df))
print(df["subject_type"].value_counts(dropna=False))

Жолдар: 77956
subject_type
1      64349
3       6978
NaN     5534
2       1095
Name: count, dtype: int64


## 3. Инфляция

Әр айдың бағасын `BASE` айының бағасына аударатын коэффициент. Тауарлар мен жұмыстарға жалпы ИПЦ, қызметтерге ақылы қызметтер индексі қолданылады.

In [3]:
infl = pd.read_excel(INFL_PATH)
infl = infl.rename(columns={infl.columns[0]: "month"})
mom_cols = list(infl.columns[1:5])
infl[mom_cols] = infl[mom_cols].apply(pd.to_numeric, errors="coerce")
infl = (infl.dropna(subset=["month"])
            .dropna(subset=mom_cols, how="all")          # әлі жарияланбаған айлар
            .sort_values("month")
            .reset_index(drop=True))

SUFFIX = BASE.strftime("%Y-%m")
for col in mom_cols:
    idx = (infl[col] / 100).cumprod()
    infl[f"{col} - {SUFFIX}"] = idx[infl["month"] == BASE].iloc[0] / idx

INFL_GOODS = f"К предыдущему месяцу товары непродо-вольственные - {SUFFIX}"
INFL_SERVICES = f"К предыдущему месяцу платные услуги - {SUFFIX}"
INFL_WORKS = f"К предыдущему месяцу платные услуги - {SUFFIX}"        # жұмыстарға бөлек индекс жоқ, жалпы ИПЦ

print(infl[["month", INFL_GOODS, INFL_SERVICES]].tail(3).to_string())

        month  К предыдущему месяцу товары непродо-вольственные - 2026-10  К предыдущему месяцу платные услуги - 2026-10
55 2026-08-01                                                    1.022121                                         1.0201
56 2026-09-01                                                    1.011000                                         1.0100
57 2026-10-01                                                    1.000000                                         1.0000


## 4. Тауарлар, қызметтер, жұмыстар

Әр санат бірдей сүзгімен таңдалады, инфляцияға түзетіледі (`unit_price_st`), ал gk медианасынан 10 есе алшақ жолдар `data_issue` болып белгіленеді: олар эталонға кірмейді, бірақ өздері бағаланады. Тауарларға қосымша `tier` (біртекті / әртекті / сирек) анықталады.

In [4]:
def prepare_segment(subject_types, infl_col):
    t = df[(df["status"] != "Не заключен")
           & df["sign_date"].notna()
           & df["unit_code"].notna()
           & (df["quantity"] > 0)
           & df["subject_type"].isin(subject_types)].copy()
    t["gk"] = t["enstru_code"] + "|" + t["unit_code"]
    t["month"] = t["sign_date"].dt.to_period("M").dt.to_timestamp()
    t = t.merge(infl[["month", infl_col]], on="month", how="left")
    t["unit_price_st"] = t["unit_price"] * t[infl_col]

    t["log_p"] = np.log(t["unit_price_st"].where(t["unit_price_st"] > 0))
    grp = t.groupby("gk")["log_p"]
    t["grp_n"], t["grp_med"] = grp.transform("count"), grp.transform("median")
    dev = t["log_p"] - t["grp_med"]
    t["data_issue"] = None
    t.loc[(t["grp_n"] >= TIER_MIN_N) & (dev >= np.log(10)), "data_issue"] = "extreme_high"
    t.loc[(t["grp_n"] >= TIER_MIN_N) & (dev <= -np.log(10)), "data_issue"] = "extreme_low"
    return t

def add_tier(t):
    clean = t[t["data_issue"].isna()]
    med_clean = clean.groupby("gk")["log_p"].median()
    abs_dev = (clean["log_p"] - clean["gk"].map(med_clean)).abs()
    stats = abs_dev.groupby(clean["gk"]).agg(n_clean="count", mad="median")

    all_dev = (t["log_p"] - t["gk"].map(med_clean)).abs()
    far_flag = (all_dev > np.log(3)).where(all_dev.notna())
    stats = stats.join(far_flag.groupby(t["gk"]).agg(n_all="count", far_share="mean"))

    by_mad = stats["mad"] > MAD_LIMIT
    by_far = (stats["n_all"] >= FAR_MIN_N) & (stats["far_share"] > FAR_LIMIT)
    stats["tier"] = np.select(
        [stats["n_all"] < TIER_MIN_N, by_mad | by_far | (stats["n_clean"] < TIER_MIN_N)],
        ["small", "heterogeneous"], default="homogeneous")

    t = t.merge(stats[["tier"]], left_on="gk", right_index=True, how="left")
    t["tier"] = t["tier"].fillna("small")
    return t

In [5]:
tovars = add_tier(prepare_segment(["1"], INFL_GOODS))
services = prepare_segment(["3"], INFL_SERVICES)
works = prepare_segment(["2"], INFL_WORKS)

for name, t in [("тауарлар", tovars), ("қызметтер", services), ("жұмыстар", works)]:
    print(f"{name:10} жолдар: {len(t):6} | gk: {t['gk'].nunique():5} | "
          f"бағасы жоқ: {t['unit_price_st'].isna().sum():4} | data_issue: {t['data_issue'].notna().sum()}")
print()
print(tovars["tier"].value_counts(normalize=True).round(3))

тауарлар   жолдар:  62980 | gk: 11020 | бағасы жоқ:    0 | data_issue: 2266
қызметтер  жолдар:   6868 | gk:   400 | бағасы жоқ:    0 | data_issue: 853
жұмыстар   жолдар:   1058 | gk:   143 | бағасы жоқ:    0 | data_issue: 104

tier
homogeneous      0.605
heterogeneous    0.265
small            0.130
Name: proportion, dtype: float64


Бұрынғы ноутбуктағыдай шығуы керек: тауарлар 62 980 жол, қызметтер 6 868 жол және 400 gk, ал `tier` шамамен 60.5% / 26.5% / 13.0%.

## 5. Әділ бағаны есептеу әдістері

`compute_ranges` — диапазон әдісі: эталон басқа ұйымдардың таза жолдарынан, басқа ұйымдар 2-ден аз болса, өз ұйымының басқа жолдарынан алынады және ескертпе қосылады. `compute_homo` — біртекті тауарлардың бұрынғы әдісі: сол gk-ның басқа шарттарындағы таза жолдардан медиана мен орташа.

In [6]:
def wquantile(x, w, qs):
    o = np.argsort(x)
    x, w = x[o], w[o]
    cw = (np.cumsum(w) - 0.5 * w) / w.sum()
    return np.interp(qs, cw, x)

def fair_range(lp, cust):
    if len(lp) == 0:
        return np.nan, np.nan, np.nan, 0, 0
    _, inv, cnt = np.unique(cust, return_inverse=True, return_counts=True)
    w = 1 / cnt[inv] if BY_CUSTOMER else np.ones(len(lp))
    p25, p50, p75 = np.exp(wquantile(lp, w, Q))
    return p25, p50, p75, len(lp), len(cnt)

def range_note(n, own):
    if pd.isna(n) or n == 0:
        return "Нет данных для сравнения"
    notes = []
    if own:
        notes.append("Данных мало: меньше 2 организаций, расчёт по закупкам этой же организации")
    if n < MIN_N:
        notes.append(f"Мало строк: расчёт по {int(n)}")
    return "; ".join(notes) or None

def compute_ranges(t):
    t = t.reset_index(drop=True).copy()
    t["lp"] = np.log(t["unit_price_st"].where(t["unit_price_st"] > 0))
    t["is_ref"] = t["data_issue"].isna() & t["lp"].notna()

    out = {}
    for gk, g in t.groupby("gk"):
        ref = g[g["is_ref"]]
        r_idx = ref.index.to_numpy()
        lp, cust = ref["lp"].to_numpy(), ref["customer_bin"].to_numpy()
        for c, gc in g.groupby("customer_bin"):
            m = cust != c
            if len(np.unique(cust[m])) >= MIN_CUST:
                res = fair_range(lp[m], cust[m])          # басқа ұйымдардан
                for i in gc.index:
                    out[i] = (*res, False)
            else:
                for i in gc.index:                         # өз жолынан басқаның бәрі
                    k = r_idx != i
                    out[i] = (*fair_range(lp[k], cust[k]), True)

    cols = ["fair_p25", "fair_med", "fair_p75", "ref_n", "ref_cust", "ref_own"]
    t = t.join(pd.DataFrame.from_dict(out, orient="index", columns=cols))
    t["fp_note"] = [range_note(n, own) for n, own in zip(t["ref_n"], t["ref_own"])]

    price = t["unit_price_st"]
    comparable = t["ref_n"] >= MIN_REF_STATUS
    t["fp_status"] = np.select(
        [t["lp"].isna(), t["fair_p25"].isna(), ~comparable,
         price < t["fair_p25"], price > t["fair_p75"]],
        ["нет цены", "нет эталона", "недостаточно данных", "ниже p25", "выше p75"],
        default="в диапазоне")
    t["over_p75"] = (price / t["fair_p75"] - 1).where(comparable & t["lp"].notna())
    return t

def homo_note(n, k):
    if pd.isna(n) or n == 0:
        return "Нет данных для сравнения"
    if n < HOMO_MIN_N or k < HOMO_MIN_CUST:
        return f"Данных мало (строк в эталоне: {int(n)}, организаций: {int(k)})"
    return None

def compute_homo(t):
    t = t.reset_index(drop=True).copy()
    t["is_ref"] = t["data_issue"].isna() & (t["unit_price_st"] > 0) & (t["quantity"] > 0)
    out = {}
    for gk, g in t.groupby("gk"):
        ref = g[g["is_ref"]]
        p = ref["unit_price_st"].to_numpy()
        con, cus = ref["contract_id"].to_numpy(), ref["customer_bin"].to_numpy()
        for c, gc in g.groupby("contract_id"):
            m = con != c
            n = int(m.sum())
            res = ((np.median(p[m]), p[m].mean(), n, len(np.unique(cus[m]))) if n
                   else (np.nan, np.nan, 0, 0))
            for i in gc.index:
                out[i] = res
    cols = ["fair_median", "fair_mean", "ref_n", "ref_cust"]
    t = t.join(pd.DataFrame.from_dict(out, orient="index", columns=cols))
    t["data_enough"] = (t["ref_n"] >= HOMO_MIN_N) & (t["ref_cust"] >= HOMO_MIN_CUST)
    t["deviation_median"] = (t["unit_price_st"] / t["fair_median"] - 1) * 100
    t["deviation_mean"] = (t["unit_price_st"] / t["fair_mean"] - 1) * 100
    t["fp_note"] = [homo_note(n, k) for n, k in zip(t["ref_n"], t["ref_cust"])]
    return t

## 6. Есептеу

### 6.1. Диапазондар

Әртекті және сирек тауарлар, қызметтер мен жұмыстар. Ең ұзақ ұяшық, бірнеше ондаған секунд жұмыс істеуі мүмкін.

In [7]:
goods_r = compute_ranges(tovars[tovars["tier"].isin(["heterogeneous", "small"])])
goods_r["category"] = np.where(goods_r["tier"] == "small", "редкие товары", "неоднородные товары")

serv_r = compute_ranges(services)
serv_r["category"] = "услуги"

works_r = compute_ranges(works)
works_r["category"] = "работы"

rng = pd.concat([goods_r, serv_r, works_r], ignore_index=True)
rng["method"] = "диапазон"
print(rng.groupby("category")["fp_status"].value_counts().unstack(fill_value=0).to_string())

fp_status            в диапазоне  выше p75  недостаточно данных  нет эталона  ниже p25
category                                                                              
неоднородные товары         5837      5271                  718            6      4867
работы                       393       284                  105           43       233
редкие товары                  0         0                 3906         4272         0
услуги                      2639      1866                  258           73      2032


Қызметтер жолында бұрынғы сандар шығуы керек: «в диапазоне» 2 639, «выше p75» 1 866, «ниже p25» 2 032, «недостаточно данных» 258, «нет эталона» 73.

### 6.2. Біртекті тауарлар

Соңғы бөлігі бірнеше жолды тікелей сүзгімен қайта есептеп, нәтиженің бірдей екенін тексереді. Барлығында «сәйкес» шығуы керек. 53480920 жолы үшін бұрынғы нәтиже: эталонда 288 жол, медиана ≈ 56 551, орташа ≈ 89 923.

In [8]:
homo_src = tovars[tovars["tier"] == "homogeneous"]
homo = compute_homo(homo_src)
homo["category"], homo["method"] = "однородные товары", "медиана"

def homo_direct(lid):
    row = homo_src.loc[homo_src["line_id"] == lid].iloc[0]
    pool = homo_src[(homo_src["gk"] == row["gk"]) & (homo_src["contract_id"] != row["contract_id"])
                    & homo_src["data_issue"].isna() & (homo_src["unit_price_st"] > 0)
                    & (homo_src["quantity"] > 0)]["unit_price_st"]
    return [pool.median(), pool.mean(), len(pool)]

ids = [i for i in [53480920] if (homo["line_id"] == i).any()]
for lid in ids + homo.sample(4, random_state=1)["line_id"].tolist():
    new = homo.loc[homo["line_id"] == lid].iloc[0]
    same = np.allclose(homo_direct(lid), [new["fair_median"], new["fair_mean"], new["ref_n"]],
                       equal_nan=True)
    print(lid, "сәйкес" if same else "СӘЙКЕС ЕМЕС", "| ref_n:", int(new["ref_n"]),
          "| медиана:", round(new["fair_median"]), "| орташа:", round(new["fair_mean"]))

53480920 сәйкес | ref_n: 288 | медиана: 56783 | орташа: 90417
48898520 сәйкес | ref_n: 8 | медиана: 46556 | орташа: 52820
50666683 сәйкес | ref_n: 60 | медиана: 224 | орташа: 257
51895797 сәйкес | ref_n: 38 | медиана: 21199 | орташа: 17571
47390967 сәйкес | ref_n: 56 | медиана: 1272 | орташа: 1260


### 6.3. Ортақ кесте `FP`

Агент тек осы кестемен жұмыс істейді. Әр жолға деңгей («выше нормы», «сильно выше»), сенімділік белгісі және бағаны сатып алу күнінің теңгесіне көшіру коэффициенті қосылады.

In [9]:
KEEP = ["line_id", "contract_id", "customer_bin", "enstru_code", "unit_code", "gk",
        "item_name_ru", "plan_extra_desc", "sign_date", "quantity", "unit_price",
        "unit_price_st", "data_issue", "category", "method",
        "fair_median", "fair_mean", "deviation_median", "deviation_mean", "data_enough",
        "fair_p25", "fair_med", "fair_p75", "over_p75", "ref_own",
        "ref_n", "ref_cust", "fp_note", "fp_status"]

FP = pd.concat([homo.reindex(columns=KEEP), rng.reindex(columns=KEEP)], ignore_index=True)

def _bin(x):
    return str(x).strip().split(".")[0].zfill(12)

FP["_lid"] = FP["line_id"].astype(str).str.strip()
FP["_bin"] = FP["customer_bin"].map(_bin)
FP["_code"] = FP["enstru_code"].astype(str).str.strip()
FP["_year"] = pd.to_datetime(FP["sign_date"], errors="coerce").dt.year
FP["_defl"] = (FP["unit_price"] / FP["unit_price_st"]).where(FP["unit_price_st"] > 0).fillna(1.0)
FP["lp"] = np.log(FP["unit_price_st"].where(FP["unit_price_st"] > 0))
FP["is_ref"] = FP["data_issue"].isna() & FP["lp"].notna() & (FP["quantity"] > 0)

# Әділ бағадан асу: біртекті тауарларда медиана немесе орташадан, диапазонда p75-тен
is_h = FP["method"] == "медиана"
dev_max = FP[["deviation_median", "deviation_mean"]].astype(float).max(axis=1) / 100
FP["excess"] = pd.to_numeric(pd.Series(np.where(is_h, dev_max, FP["over_p75"]), index=FP.index),
                             errors="coerce")

FP.loc[is_h, "fp_status"] = np.select(
    [FP.loc[is_h, "ref_n"].fillna(0) == 0,
     FP.loc[is_h, "excess"] > OVER,
     FP.loc[is_h, "excess"] < -OVER],
    ["нет эталона", "выше справедливой", "ниже справедливой"],
    default="в пределах нормы")

FP["fp_level"] = np.select([FP["excess"] >= HIGH_X - 1, FP["excess"] > OVER],
                           ["сильно выше", "выше нормы"], default="")
FP["fp_level"] = FP["fp_level"].mask(FP["fp_level"] == "")
FP["reliable"] = np.where(is_h, FP["data_enough"].fillna(False).astype(bool), FP["fp_note"].isna())

ALL_LIDS = set(df["line_id"].astype(str))
CATEGORIES = ["однородные товары", "неоднородные товары", "редкие товары", "услуги", "работы"]

print(FP.groupby("category").agg(
    строк=("line_id", "size"), gk=("gk", "nunique"),
    есть_эталон=("ref_n", lambda s: round((s.fillna(0) > 0).mean(), 3)),
    выше_нормы=("fp_level", lambda s: int((s == "выше нормы").sum())),
    сильно_выше=("fp_level", lambda s: int((s == "сильно выше").sum()))).to_string())

                     строк    gk  есть_эталон  выше_нормы  сильно_выше
category                                                              
неоднородные товары  16699   983        1.000        2051         2060
однородные товары    38103  3837        0.984        8688         1965
работы                1058   143        0.959         102          120
редкие товары         8178  6200        0.478           0            0
услуги                6868   400        0.989         716          771


## 7. Көлем

### 7.1. Жоспар деректері

Көлем жылдық жоспардан (`plan_items`) алынады, өйткені 2026 жылғы шарттар әлі толық емес. Әр позицияның тек ағымдағы нұсқасы қолданылады: `is_current` бағаны болса соған, болмаса `source = 'plan'` және өшірілмеген жолдарға сүйенеміз.

In [10]:
pcols = set(pd.read_sql(
    "SELECT column_name FROM information_schema.columns WHERE table_name = 'plan_items'",
    pg)["column_name"])
CURRENT = ("is_current::text IN ('true', '1')" if "is_current" in pcols
           else "source = 'plan' AND COALESCE(is_deleted::text, '0') IN ('0', 'false')")
print("Ағымдағы нұсқа сүзгісі:", CURRENT)

plan = pd.read_sql(f"""
    SELECT plan_id, customer_bin, year, enstru_code, unit_code, subject_type,
           name_ru, quantity, amount
    FROM plan_items
    WHERE {CURRENT}
      AND quantity > 0
      AND enstru_code IS NOT NULL AND enstru_code <> ''
""", pg)
print("Жоспар жолдары:", len(plan))

Ағымдағы нұсқа сүзгісі: is_current::text IN ('true', '1')
Жоспар жолдары: 73959


In [11]:
plan["subject_type"] = plan["subject_type"].astype(str).str.replace(r"\.0$", "", regex=True)
plan["category"] = plan["subject_type"].map({"1": "товары", "2": "работы", "3": "услуги"})
plan["unit_code"] = plan["unit_code"].astype(str).str.replace(r"\.0$", "", regex=True)
plan["year"] = pd.to_numeric(plan["year"], errors="coerce")
plan["quantity"] = pd.to_numeric(plan["quantity"], errors="coerce")
plan["amount"] = pd.to_numeric(plan["amount"], errors="coerce").fillna(0)
plan = plan[plan["category"].notna() & plan["year"].notna() & (plan["quantity"] > 0)].copy()
plan["year"] = plan["year"].astype(int)
plan["gk"] = plan["enstru_code"].astype(str).str.strip() + "|" + plan["unit_code"]

year_n = plan["year"].value_counts().sort_index()
FIRST_YEAR = int(year_n[year_n >= 100].index.min())     # деректер басталатын жыл
plan = plan[plan["year"] >= FIRST_YEAR]

print("Деректер басталатын жыл:", FIRST_YEAR, "\n")
print("Жолдар саны:")
print(plan.pivot_table(index="year", columns="category", values="amount",
                       aggfunc="size", fill_value=0).to_string(), "\n")
print("Сома, млн теңге:")
print((plan.pivot_table(index="year", columns="category", values="amount",
                        aggfunc="sum", fill_value=0) / 1e6).round(0).to_string())

Деректер басталатын жыл: 2024 

Жолдар саны:
category  работы  товары  услуги
year                            
2024         427   24253    2927
2025         410   22814    2533
2026         269   18364    1957 

Сома, млн теңге:
category  работы   товары   услуги
year                              
2024      5274.0  34516.0  54139.0
2025      5928.0  23909.0  67111.0
2026      4268.0  18664.0  61328.0


Жылдар бойынша сома жоспардың толықтығын көрсетеді: 2026 жылғы сома алдыңғы жылдардан күрт аз болса, оны салыстыруда сақ болу керек.

### 7.2. Өсуді есептеу

Әр ұйым × позиция (ЕНС ТРУ + өлшем бірлігі) × жыл бойынша сан қосылады. База — алдыңғы жылдардың орташа саны, өсу — биылғы сан / база. Ұйымның «әдеттегі өзгерісі» — сол жылы оның басқа позицияларының өсуінің p25–p75 диапазоны (позицияның өзі есептен шығарылады). Саны тұрақты 1 болатын позициялар (көбіне қызметтер мен жұмыстар) бұл диапазонға кірмейді, өйткені олардың саны ештеңе айтпайды.

Позиция «нетипичный рост» болып белгіленеді, егер сан кемінде `VOL_MIN_GROWTH` есе өссе **және** ұйымның әдеттегі өзгерісінің p75-інен `VOL_ORG_X` есе жоғары болса. Ұйымда салыстыратын позиция аз болса, екінші шарт қолданылмайды, бірақ ескертпе қосылады.

In [12]:
VOL = (plan.groupby(["customer_bin", "gk", "year"], as_index=False)
           .agg(enstru_code=("enstru_code", "first"), unit_code=("unit_code", "first"),
                category=("category", "first"), name=("name_ru", "first"),
                qty=("quantity", "sum"), amount=("amount", "sum"),
                plan_lines=("plan_id", "size"))
           .sort_values(["customer_bin", "gk", "year"])
           .reset_index(drop=True))

# База: алдыңғы жылдардың орташа саны
g = VOL.groupby(["customer_bin", "gk"])
VOL["prev_years"] = g.cumcount()
VOL["base"] = ((g["qty"].cumsum() - VOL["qty"]) / VOL["prev_years"]).where(VOL["prev_years"] > 0)
VOL["growth"] = VOL["qty"] / VOL["base"]
VOL["lg"] = np.log(VOL["growth"])
VOL["extra_qty"] = VOL["qty"] - VOL["base"]
VOL["extra_amount"] = VOL["extra_qty"] * VOL["amount"] / VOL["qty"]   # тек сұрыптау үшін
VOL["const_one"] = (VOL["qty"] == 1) & (VOL["base"] == 1)

# Ұйымның әдеттегі өзгерісі: басқа позицияларының өсуі, p25–p75
VOL["org_p25"], VOL["org_p75"], VOL["org_n"] = np.nan, np.nan, 0
inform = VOL["base"].notna() & ~VOL["const_one"]
for _, grp in VOL[inform].groupby(["customer_bin", "year"]):
    lg, n = grp["lg"].to_numpy(), len(grp)
    if n < 2:
        continue
    if n > 200:                                   # үлкен топта позицияның өз әсері елеусіз
        P = np.tile(np.quantile(lg, [0.25, 0.75]), (n, 1))
    else:
        P = np.array([np.quantile(np.delete(lg, i), [0.25, 0.75]) for i in range(n)])
    VOL.loc[grp.index, ["org_p25", "org_p75"]] = np.exp(P)
    VOL.loc[grp.index, "org_n"] = n - 1

# Деңгей мен статус
has_hist = VOL["base"].notna()
org_ok = VOL["org_n"] >= VOL_MIN_ORG_ITEMS
atypical = ~org_ok | (VOL["growth"] >= VOL["org_p75"] * VOL_ORG_X)
cand = has_hist & ~VOL["const_one"] & atypical
VOL["vol_level"] = np.select(
    [cand & (VOL["growth"] >= VOL_HIGH), cand & (VOL["growth"] >= VOL_MIN_GROWTH)],
    ["сильно выше", "выше обычного"], default="")
VOL["vol_level"] = VOL["vol_level"].mask(VOL["vol_level"] == "")

VOL["vol_status"] = np.select(
    [~has_hist & (VOL["year"] == FIRST_YEAR), ~has_hist, VOL["const_one"],
     VOL["vol_level"].notna(), VOL["growth"] > 1, VOL["growth"] < 1],
    ["нет данных за прошлые годы", "новая позиция", "количество не меняется (1 ед.)",
     "нетипичный рост", "рост в пределах обычного", "снижение"],
    default="без изменений")

def vol_note(base, year, const_one, category, prev_years, org_n):
    if pd.isna(base):
        return ("Данные начинаются с этого года: сравнивать не с чем" if year == FIRST_YEAR
                else "Позиция появилась впервые: сравнивать не с чем")
    notes = []
    if const_one and category != "товары":
        notes.append("Количество указано как 1 ед.: объём услуги или работы в количестве не отражается")
    if prev_years == 1:
        notes.append("История только за один предыдущий год")
    if not const_one and org_n < VOL_MIN_ORG_ITEMS:
        notes.append("У организации мало позиций для сравнения: нетипичность оценена "
                     "только по собственной истории позиции")
    return "; ".join(notes) or None

VOL["vol_note"] = [vol_note(*a) for a in zip(VOL["base"], VOL["year"], VOL["const_one"],
                                             VOL["category"], VOL["prev_years"], VOL["org_n"])]
VOL["_bin"] = VOL["customer_bin"].map(_bin)
VOL["_code"] = VOL["enstru_code"].astype(str).str.strip()
VOL_CATEGORIES = ["товары", "услуги", "работы"]
print("Позициялар (ұйым × gk × жыл):", len(VOL))

Позициялар (ұйым × gk × жыл): 36008


### 7.3. Тексеру және шектерді таңдау

Бірінші кесте әр жыл мен санат бойынша статустарды көрсетеді. «Нетипичный рост» бағаланатын позициялардың шамамен 5–10%-ы болғаны дұрыс: көп болса, `VOL_MIN_GROWTH` немесе `VOL_ORG_X`-ті көтеріңіз. Екінші кесте ұйымдардың әдеттегі өзгерісі қаншалықты кең екенін, ал үшіншісі өсудің жалпы таралуын көрсетеді.

In [13]:
print(VOL.groupby(["year", "category"])["vol_status"].value_counts()
         .unstack(fill_value=0).to_string(), "\n")

o = VOL[VOL["org_n"] > 0].drop_duplicates(["customer_bin", "year"])
print("Ұйымдардың әдеттегі өзгерісі (p25–p75), ұйым × жыл бойынша:")
print(o[["org_n", "org_p25", "org_p75"]].describe(percentiles=[.25, .5, .75]).round(2), "\n")

ev = VOL[VOL["base"].notna() & ~VOL["const_one"]]
print(f"Бағаланатын позициялар: {len(ev)} | нетипичный рост: {VOL['vol_level'].notna().sum()} "
      f"({VOL['vol_level'].notna().sum() / max(len(ev), 1):.1%})")
print("Өсудің таралуы:")
print(ev["growth"].quantile([.5, .75, .9, .95, .99]).round(2))

vol_status     без изменений  количество не меняется (1 ед.)  нет данных за прошлые годы  нетипичный рост  новая позиция  рост в пределах обычного  снижение
year category                                                                                                                                               
2024 работы                0                               0                         212                0              0                         0         0
     товары                0                               0                       11810                0              0                         0         0
     услуги                0                               0                        1222                0              0                         0         0
2025 работы               13                              36                           0               18            144                         8        31
     товары              676                              

## 8. Агенттің құралдары

### 8.1. Баға

`fair_price` әділ бағаны береді: шарт жолы бойынша (сатып алу күнінің теңгесінде) немесе ЕНС ТРУ коды бойынша (`BASE` айының бағасында). `find_anomalies` бағасы әділ бағадан асқан жолдарды санаттарға бөліп қайтарады.

In [14]:
def _num(x, nd=0):
    if x is None or pd.isna(x):
        return None
    return int(round(float(x))) if nd == 0 else round(float(x), nd)

def _s(x, n=None):
    if x is None or pd.isna(x):
        return None
    s = str(x).strip()
    return s[:n] if n else s

def _line_view(d):
    rows = []
    for _, r in d.iterrows():
        k = r["_defl"]
        row = {"line_id": r["_lid"], "category": r["category"],
               "contract_id": _s(r["contract_id"]), "customer_bin": r["_bin"],
               "item": _s(r["item_name_ru"]), "description": _s(r["plan_extra_desc"], 150),
               "sign_date": None if pd.isna(r["sign_date"])
                            else pd.Timestamp(r["sign_date"]).strftime("%Y-%m-%d"),
               "unit": _s(r["unit_code"]), "quantity": _num(r["quantity"], 2),
               "unit_price": _num(r["unit_price"])}
        if r["method"] == "медиана":
            row.update({"fair_median": _num(r["fair_median"] * k),
                        "fair_mean": _num(r["fair_mean"] * k),
                        "deviation_median_pct": _num(r["deviation_median"], 1),
                        "deviation_mean_pct": _num(r["deviation_mean"], 1),
                        "data_enough": bool(r["data_enough"])})
        else:
            row.update({"fair_p25": _num(r["fair_p25"] * k),
                        "fair_med": _num(r["fair_med"] * k),
                        "fair_p75": _num(r["fair_p75"] * k),
                        "over_p75_pct": _num(r["over_p75"] * 100),
                        "times_p75": _num(r["over_p75"] + 1, 1),
                        "ref_own_org": None if pd.isna(r["ref_own"]) else bool(r["ref_own"])})
        row.update({"status": r["fp_status"], "level": _s(r["fp_level"]),
                    "note": _s(r["fp_note"]),
                    "ref_lines": _num(r["ref_n"]), "ref_orgs": _num(r["ref_cust"])})
        rows.append(row)
    return rows

def _gk_range(g, customer_bin=None):
    ref = g[g["is_ref"]]
    own_used = False
    if customer_bin:
        others = ref[ref["_bin"] != _bin(customer_bin)]
        if others["_bin"].nunique() >= MIN_CUST:
            ref = others
        else:
            own_used = bool((ref["_bin"] == _bin(customer_bin)).any())
    p25, p50, p75, n, k = fair_range(ref["lp"].to_numpy(), ref["_bin"].to_numpy())
    notes = []
    if n == 0:
        notes.append("Нет данных для сравнения")
    else:
        if k < MIN_CUST or own_used:
            notes.append("Данных мало: меньше 2 организаций"
                         + (", расчёт с учётом закупок этой же организации" if own_used else ""))
        if n < MIN_N:
            notes.append(f"Мало строк: расчёт по {n}")
    return {"fair_p25": _num(p25), "fair_med": _num(p50), "fair_p75": _num(p75),
            "ref_lines": int(n), "ref_orgs": int(k), "reliable": bool(n >= MIN_REF_STATUS),
            "note": "; ".join(notes) or None}

def _gk_median(g, customer_bin=None):
    ref = g[g["is_ref"]]
    notes = []
    if customer_bin:
        others = ref[ref["_bin"] != _bin(customer_bin)]
        if len(others):
            ref = others
        else:
            notes.append("Других организаций нет, расчёт по закупкам этой же организации")
    p = ref["unit_price_st"]
    n, k = len(p), int(ref["_bin"].nunique())
    if n == 0:
        notes = ["Нет данных для сравнения"]
    elif n < HOMO_MIN_N or k < HOMO_MIN_CUST:
        notes.append(f"Данных мало (строк в эталоне: {n}, организаций: {k})")
    return {"fair_median": _num(p.median()), "fair_mean": _num(p.mean()),
            "ref_lines": n, "ref_orgs": k,
            "reliable": bool(n >= HOMO_MIN_N and k >= HOMO_MIN_CUST),
            "note": "; ".join(notes) or None}

def fair_price(line_id=None, enstru_code=None, customer_bin=None):
    """Справедливая цена: по строке договора или по коду ЕНСТРУ."""
    if line_id is not None and str(line_id).strip():
        lid = str(line_id).strip().split(".")[0]
        d = FP[FP["_lid"] == lid]
        if d.empty:
            msg = ("Строка есть в базе, но не участвует в анализе: договор не заключён, "
                   "нет даты, цены, единицы измерения или привязки к плану."
                   if lid in ALL_LIDS else f"Строка договора {lid} не найдена.")
            return {"status": "not_found", "message": msg}
        return {"status": "ok", "mode": "line", "price_basis": "тенге на дату закупки",
                "line": _line_view(d)[0]}

    if enstru_code:
        code = str(enstru_code).strip()
        d = FP[FP["_code"] == code]
        if d.empty:
            return {"status": "not_found", "message": f"Закупки по ЕНСТРУ {code} не найдены."}
        results = []
        for (gk, cat), g in d.groupby(["gk", "category"]):
            method = g["method"].iloc[0]
            calc = _gk_median if method == "медиана" else _gk_range
            results.append({"unit": _s(g["unit_code"].iloc[0]), "category": cat,
                            "method": method, "lines_total": len(g), **calc(g, customer_bin)})
        names = d["item_name_ru"].dropna()
        return {"status": "ok", "mode": "enstru", "enstru_code": code,
                "item": names.mode().iloc[0] if len(names) else None,
                "price_basis": PRICE_BASIS,
                "excluded_customer": _bin(customer_bin) if customer_bin else None,
                "results": results}

    return {"status": "error", "message": "Нужно указать line_id или enstru_code."}

def find_anomalies(category=None, enstru_code=None, customer_bin=None, year=None,
                   threshold=OVER, level="выше нормы", limit=10):
    """Закупки с ценой выше справедливой, по категориям."""
    thr = float(threshold)
    if thr > 1:                                  # модель 30 деп жіберсе
        thr /= 100
    strong = level == "сильно выше"
    limit = max(1, min(int(limit), MAX_LIMIT))

    d, filters = FP, {}
    if category:
        filters["category"] = category
        cats = CATEGORIES[:3] if category == "товары" else [category]
        d = d[d["category"].isin(cats)]
    if enstru_code:
        filters["enstru_code"] = str(enstru_code).strip()
        d = d[d["_code"] == filters["enstru_code"]]
    if customer_bin:
        filters["customer_bin"] = _bin(customer_bin)
        d = d[d["_bin"] == filters["customer_bin"]]
    if year:
        filters["year"] = int(year)
        d = d[d["_year"] == filters["year"]]
    if d.empty:
        return {"status": "not_found", "filters": filters,
                "message": "По заданным условиям закупки не найдены.",
                "available_categories": CATEGORIES}

    groups, total = [], 0
    for cat in CATEGORIES:
        sub = d[d["category"] == cat]
        if sub.empty:
            continue
        method = sub["method"].iloc[0]
        comp = sub[sub["excess"].notna()]
        hit = comp[comp["excess"] >= HIGH_X - 1] if strong else comp[comp["excess"] > thr]
        hit = hit.sort_values(["reliable", "excess"], ascending=[False, False])
        base = ("медианы или средней цены других договоров" if method == "медиана"
                else "верхней границы справедливого диапазона (p75)")
        rule = (f"цена в {HIGH_X:g} и более раз выше {base}" if strong
                else f"цена выше {base} более чем на {thr:.0%}")
        groups.append({
            "category": cat, "method": method, "rule": rule,
            "lines_total": len(sub), "lines_comparable": len(comp),
            "share_over_norm_pct": _num((comp["excess"] > thr).mean() * 100, 1) if len(comp) else None,
            "lines_found": len(hit), "lines_found_reliable": int(hit["reliable"].sum()),
            "customers_found": int(hit["_bin"].nunique()),
            "rows": _line_view(hit.head(limit)), "truncated": len(hit) > limit})
        total += len(hit)
    return {"status": "ok", "filters": filters, "level": level,
            "total_found": total, "groups": groups}

### 8.2. Көлем

`volume_check` бір ұйымның бір позициясы бойынша санның жылдар бойынша өзгерісін және бағасын береді. `find_volume_anomalies` нетипичный рост табылған позицияларды артық сома бойынша сұрыптап қайтарады.

In [15]:
def _vol_row(r):
    return {"customer_bin": r["_bin"], "year": int(r["year"]), "category": r["category"],
            "enstru_code": r["_code"], "unit": _s(r["unit_code"]), "item": _s(r["name"]),
            "qty": _num(r["qty"], 2), "base_qty": _num(r["base"], 2),
            "prev_years": int(r["prev_years"]), "growth_x": _num(r["growth"], 2),
            "org_typical_from_x": _num(r["org_p25"], 2), "org_typical_to_x": _num(r["org_p75"], 2),
            "org_positions_compared": int(r["org_n"]),
            "extra_qty": _num(r["extra_qty"], 2), "extra_amount": _num(r["extra_amount"]),
            "status": r["vol_status"], "level": _s(r["vol_level"]), "note": _s(r["vol_note"])}

def volume_check(customer_bin, enstru_code, year=None):
    """Динамика количества одной позиции у организации."""
    b, code = _bin(customer_bin), str(enstru_code).strip()
    d = VOL[(VOL["_bin"] == b) & (VOL["_code"] == code)]
    if d.empty:
        return {"status": "not_found",
                "message": f"В планах организации {b} позиция ЕНСТРУ {code} не найдена."}
    results = []
    for gk, g in d.groupby("gk"):
        g = g.sort_values("year")
        cur = g[g["year"] == int(year)] if year else g.tail(1)
        results.append({
            "unit": _s(g["unit_code"].iloc[0]),
            "history": [{"year": int(y), "qty": _num(q, 2), "amount": _num(a)}
                        for y, q, a in zip(g["year"], g["qty"], g["amount"])],
            "evaluation": _vol_row(cur.iloc[0]) if len(cur) else None})
    return {"status": "ok", "customer_bin": b, "enstru_code": code,
            "item": _s(d["name"].iloc[0]), "results": results}

def find_volume_anomalies(category=None, customer_bin=None, enstru_code=None, year=None,
                          level="выше обычного", limit=10):
    """Нетипичный рост количества по сравнению с предыдущими годами."""
    limit = max(1, min(int(limit), MAX_LIMIT))
    d, filters = VOL, {}
    if category:
        filters["category"] = category
        d = d[d["category"] == category]
    if customer_bin:
        filters["customer_bin"] = _bin(customer_bin)
        d = d[d["_bin"] == filters["customer_bin"]]
    if enstru_code:
        filters["enstru_code"] = str(enstru_code).strip()
        d = d[d["_code"] == filters["enstru_code"]]
    if year:
        filters["year"] = int(year)
        d = d[d["year"] == filters["year"]]
    if d.empty:
        return {"status": "not_found", "filters": filters,
                "message": "По заданным условиям позиции в планах не найдены.",
                "available_categories": VOL_CATEGORIES}

    strong = level == "сильно выше"
    hit = d[d["vol_level"] == "сильно выше"] if strong else d[d["vol_level"].notna()]
    hit = hit.sort_values("extra_amount", ascending=False)
    ev = d["base"].notna() & ~d["const_one"]
    rule = (f"количество выросло в {VOL_HIGH:g} и более раз по сравнению со средним "
            "за предыдущие годы, и это нетипично для организации" if strong else
            f"количество выросло минимум в {VOL_MIN_GROWTH:g} раза по сравнению со средним "
            f"за предыдущие годы и более чем на {VOL_ORG_X - 1:.0%} превышает обычное "
            "изменение других позиций этой организации")
    summary = {
        "filters": filters, "level": level, "rule": rule,
        "years": sorted(int(y) for y in d["year"].unique()),
        "positions_total": len(d),
        "positions_evaluated": int(ev.sum()),
        "positions_without_history": int(d["base"].isna().sum()),
        "positions_qty_always_1": int(d["const_one"].sum()),
        "found": len(hit),
        "found_strong": int((hit["vol_level"] == "сильно выше").sum()),
        "share_found_pct": _num(len(hit) / ev.sum() * 100, 1) if ev.sum() else None,
        "found_by_category": {k: int(v) for k, v in hit["category"].value_counts().items()},
        "customers_found": int(hit["_bin"].nunique()),
        "extra_amount_total": _num(hit["extra_amount"].sum()),
    }
    return {"status": "ok", "summary": summary,
            "rows": [_vol_row(r) for _, r in hit.head(limit).iterrows()],
            "truncated": len(hit) > limit}

## 9. Құралдарды модельсіз тексеру

API шақырылмайды, токен жұмсалмайды. Қате шықса, агентке өтпей тұрып, алдымен соны түзетеміз.

In [16]:
def show(r, n=2500):
    print(json.dumps(r, ensure_ascii=False, indent=2)[:n], "\n" + "─" * 80)

homo_lines = FP.loc[FP["category"] == "однородные товары", "_lid"]
lid_homo = "53480920" if (homo_lines == "53480920").any() else homo_lines.iloc[0]
code_homo = ("265152.790.000057" if (FP["_code"] == "265152.790.000057").any()
             else FP.loc[FP["category"] == "однородные товары", "_code"].value_counts().index[0])
srv = FP[FP["category"] == "услуги"]
lid_srv = srv.loc[srv["fp_level"] == "сильно выше", "_lid"].iloc[0] \
          if (srv["fp_level"] == "сильно выше").any() else srv["_lid"].iloc[0]
code_het = FP.loc[FP["category"] == "неоднородные товары", "_code"].value_counts().index[0]
bin_top = FP.loc[FP["fp_level"].notna(), "_bin"].value_counts().index[0]

show(fair_price(line_id=lid_homo))
show(fair_price(line_id=lid_srv))
show(fair_price(enstru_code=code_het))

r = find_anomalies(customer_bin=bin_top, limit=2)
show({"total_found": r["total_found"],
      "groups": [{k: v for k, v in g.items() if k != "rows"} for g in r["groups"]]})

vt = VOL[VOL["vol_level"].notna()].sort_values("extra_amount", ascending=False)
vol_bin, vol_code = vt.iloc[0]["_bin"], vt.iloc[0]["_code"]
last_year = int(VOL["year"].max())

show(volume_check(vol_bin, vol_code))
r = find_volume_anomalies(year=last_year, limit=2)
show({"summary": r["summary"], "first_row": r["rows"][:1]})

{
  "status": "ok",
  "mode": "line",
  "price_basis": "тенге на дату закупки",
  "line": {
    "line_id": "53480920",
    "category": "однородные товары",
    "contract_id": "23554778",
    "customer_bin": "990740002243",
    "item": "Калибратор",
    "description": "Калибратор C-Peptide CalSet для анализатора Cobas PRO",
    "sign_date": "2025-10-03",
    "unit": "778",
    "quantity": 1.0,
    "unit_price": 56000,
    "fair_median": 51249,
    "fair_mean": 81605,
    "deviation_median_pct": 9.3,
    "deviation_mean_pct": -31.4,
    "data_enough": true,
    "status": "в пределах нормы",
    "level": null,
    "note": null,
    "ref_lines": 288,
    "ref_orgs": 2
  }
} 
────────────────────────────────────────────────────────────────────────────────
{
  "status": "ok",
  "mode": "line",
  "price_basis": "тенге на дату закупки",
  "line": {
    "line_id": "53608351",
    "category": "услуги",
    "contract_id": "23618992",
    "customer_bin": "940940000384",
    "item": "Услуги по прод

## 10. Агент

### 10.1. Модель, құралдардың сипаттамасы және нұсқаулық

Модель құрал мен параметрлерді сипаттама бойынша таңдайды, ал `SYSTEM` жауапты қалай жазу керегін айтады. Провайдер мен модель `.env`-тен алынады (`LLM_PROVIDER`, `LLM_MODEL`).

In [52]:
if os.getenv("LLM_PROVIDER", "openai") == "gemini":
    client = OpenAI(api_key=os.getenv("GEMINI_API_KEY"),
                    base_url="https://generativelanguage.googleapis.com/v1beta/openai/")
    MODEL = os.getenv("LLM_MODEL", "gemini-2.5-flash")
else:
    client = OpenAI()
    MODEL = os.getenv("LLM_MODEL", "gpt-5.4-mini")

OAI_TOOLS = [
    {"type": "function", "function": {
        "name": "fair_price",
        "description": (
            "Определяет справедливую цену. По line_id — для конкретной строки договора: "
            "категория, фактическая цена, справедливая цена (для однородных товаров — медиана "
            "и средняя других договоров, для остальных категорий — диапазон p25–p75), "
            "отклонение, статус и предупреждения. По enstru_code — справедливая цена по коду "
            "ЕНСТРУ отдельно для каждой единицы измерения; если указан customer_bin, закупки "
            "этой организации исключаются из расчёта. Укажи line_id или enstru_code."),
        "parameters": {"type": "object", "properties": {
            "line_id": {"type": "string", "description": "ID строки договора"},
            "enstru_code": {"type": "string",
                            "description": "Код ЕНСТРУ, например '222926.150.000002'"},
            "customer_bin": {"type": "string", "description": "БИН организации, 12 цифр"}}}}},
    {"type": "function", "function": {
        "name": "find_anomalies",
        "description": (
            "Ищет закупки с ценой выше справедливой по всем категориям: однородные товары, "
            "неоднородные товары, редкие товары, услуги, работы. Результат разбит по категориям. "
            "Все фильтры необязательны. threshold — порог как доля (30% = 0.30), по умолчанию 0.30. "
            f"level='сильно выше' — только цены в {HIGH_X:g} и более раз выше справедливой; "
            "используй его, если просят самые сильные, крупные или явные завышения."),
        "parameters": {"type": "object", "properties": {
            "category": {"type": "string", "enum": ["товары"] + CATEGORIES,
                         "description": "Категория; 'товары' — все три категории товаров"},
            "enstru_code": {"type": "string", "description": "Код ЕНСТРУ"},
            "customer_bin": {"type": "string", "description": "БИН организации, 12 цифр"},
            "year": {"type": "integer", "description": "Год заключения договора"},
            "threshold": {"type": "number", "description": "Порог как доля: 30% = 0.30"},
            "level": {"type": "string", "enum": ["выше нормы", "сильно выше"]},
            "limit": {"type": "integer", "minimum": 1, "maximum": MAX_LIMIT,
                      "description": "Сколько строк вернуть по каждой категории, по умолчанию 10"}}}}},
    {"type": "function", "function": {
        "name": "volume_check",
        "description": (
            "Показывает, как менялось количество одной позиции (код ЕНСТРУ) в годовых планах "
            "организации по годам, и оценивает рост за выбранный год (по умолчанию последний): "
            "среднее за предыдущие годы, во сколько раз выросло количество, обычное изменение "
            "других позиций этой организации, статус и предупреждения."),
        "parameters": {"type": "object", "properties": {
            "customer_bin": {"type": "string", "description": "БИН организации, 12 цифр"},
            "enstru_code": {"type": "string", "description": "Код ЕНСТРУ"},
            "year": {"type": "integer", "description": "Год плана, по умолчанию последний"}},
            "required": ["customer_bin", "enstru_code"]}}},
    {"type": "function", "function": {
        "name": "find_volume_anomalies",
        "description": (
            "Выявляет нетипичное завышение количества ТРУ по сравнению с предыдущими годами "
            "по годовым планам закупок: товары, услуги и работы. Количество позиции сравнивается "
            "со средним за предыдущие годы той же организации, а нетипичность — с тем, как в этом "
            "году изменились остальные позиции организации. Все фильтры необязательны. "
            "Результаты отсортированы по стоимости дополнительного количества. "
            f"level='сильно выше' — только рост в {VOL_HIGH:g} и более раз."),
        "parameters": {"type": "object", "properties": {
            "category": {"type": "string", "enum": VOL_CATEGORIES},
            "customer_bin": {"type": "string", "description": "БИН организации, 12 цифр"},
            "enstru_code": {"type": "string", "description": "Код ЕНСТРУ"},
            "year": {"type": "integer", "description": "Год плана"},
            "level": {"type": "string", "enum": ["выше обычного", "сильно выше"]},
            "limit": {"type": "integer", "minimum": 1, "maximum": MAX_LIMIT,
                      "description": "Сколько позиций вернуть, по умолчанию 10"}}}}},
]

SYSTEM = f"""Ты аналитик государственных закупок Казахстана.
Всегда отвечай на том же языке, на котором задается вопрос. Никогда не отвечай на другом языке.
Все числа бери только из результатов инструментов, ничего не придумывай и не пересчитывай.
Различай вопросы о цене и о количестве: цена — инструменты fair_price и find_anomalies,
количество (объёмы) — инструменты volume_check и find_volume_anomalies.

ЦЕНА

Справедливая цена зависит от категории (поле category):
1. Однородные товары (method = «медиана»): справедливая цена — медиана и средняя цена того же
   товара (тот же ЕНСТРУ и единица измерения) в других договорах. Отклонения уже в процентах.
   Строки с data_enough = false ненадёжны, перечисляй их отдельно.
2. Неоднородные товары, редкие товары, услуги и работы (method = «диапазон»): цены внутри
   одного кода сильно различаются, поэтому справедливая цена — диапазон от p25 до p75
   по закупкам других организаций, а не одно число. Всегда называй диапазон,
   медиану приводи только как ориентир. Редкие товары — товары с очень малым числом закупок.

Цены:
- По строке договора все цены (фактическая и справедливая) указаны в тенге на дату закупки,
  их можно сравнивать напрямую.
- По коду ЕНСТРУ справедливая цена указана в ценах из поля price_basis.

Предупреждения:
- Если поле note не пустое, обязательно передай его смысл пользователю.
- Если ref_own_org = true, эталон построен по закупкам этой же организации: говори
  «цена отличается (не отличается) от других закупок этой организации», а не «цена справедливая».
- Статус «недостаточно данных»: диапазон только для справки, вывод о завышении не делай.
- Статус «нет эталона»: скажи, что сравнить не с чем.

Аномалии:
- Однородные товары: цена выше медианы или средней других договоров больше чем на порог.
- Категории с диапазоном: «выше нормы» — цена выше p75 больше чем на порог (по умолчанию
  {OVER:.0%}, требование ТЗ). Таких строк много (поле share_over_norm_pct), это сигнал для
  просмотра, а не доказательство. «сильно выше» — цена в {HIGH_X:g} и более раз выше
  справедливой, это приоритет для проверки.
- Цена услуг и работ часто зависит от объёма и срока, которые не указаны в описании.
  Не называй превышение переплатой или нарушением: это повод проверить договор.

КОЛИЧЕСТВО (ОБЪЁМЫ)
- Источник — годовые планы закупок. Количество позиции (ЕНСТРУ и единица измерения)
  сравнивается со средним за предыдущие годы той же организации (base_qty);
  growth_x — во сколько раз выросло количество.
- Обычное изменение для организации — диапазон от org_typical_from_x до org_typical_to_x:
  так в этом году изменились остальные позиции этой же организации (p25–p75).
- Нетипичный рост («выше обычного»): количество выросло минимум в {VOL_MIN_GROWTH:g} раза
  и более чем на {VOL_ORG_X - 1:.0%} превышает верхнюю границу обычного изменения
  организации. «сильно выше» — рост в {VOL_HIGH:g} и более раз.
- Аномалия определяется только по количеству. extra_amount — стоимость дополнительного
  количества по цене текущего года, она нужна только для порядка: сначала самые дорогие случаи.
- Новые позиции и позиции с количеством 1 ед. оценить нельзя, так и скажи. Если note не
  пустое, передай его смысл. Если evaluation = null, в этом году позиции в плане нет.
- Рост количества — повод проверить обоснованность потребности, а не доказательство нарушения.

Стиль ответа:
- Начинай с краткого вывода (1–3 предложения) простым языком, без названий полей
  и без больших таблиц.
- Если найдено по нескольким категориям, коротко скажи о каждой отдельно.
- Если truncated = true, назови общее число найденных строк, скажи, что показаны самые
  сильные отклонения, а полный список можно выгрузить.
- Если без кода ЕНСТРУ, строки договора или БИН ответить нельзя, коротко уточни.
- Отвечай коротко , 1-3 предложениями.
- Отвечай максимально доступно , как пользователю сервиса, как человеку который не видел всю разработку и названия переменных.
- При ответе не приводи ниже данные, в стиле как 'ниже приведены результаты в таблице', или 'вы можете скачать полный список'. Давай весь резюмированный ответ сразу в самом тексте. 
- В ответе не называй товары одновродными или неоднородными. Потому что однородные товары - это просто товары, цены которых не сильно разбросаны в своей группе. А неоднородные - сильно разбросаны. При ответе переформулируй.
- Если вопрос задается на казахском - отвечай на казахском.
"""

TOOL_FUNCS = {"fair_price": fair_price, "find_anomalies": find_anomalies,
              "volume_check": volume_check, "find_volume_anomalies": find_volume_anomalies}
for t in OAI_TOOLS:
    assert t["function"]["name"] in TOOL_FUNCS
print("Инструменты:", list(TOOL_FUNCS), "| модель:", MODEL)

Инструменты: ['fair_price', 'find_anomalies', 'volume_check', 'find_volume_anomalies'] | модель: gpt-5.4-mini


### 10.2. Агент циклі

Модель құрал шақырса, біз оны орындап, нәтижесін қайтарамыз. Модель құрал шақыруды тоқтатқанда, оның мәтіні соңғы жауап болады.

In [53]:
def run_tool(name, args_json):
    try:
        args = {k: v for k, v in json.loads(args_json or "{}").items() if v not in (None, "")}
        result = TOOL_FUNCS[name](**args)
    except Exception as e:
        result = {"status": "error", "message": f"{type(e).__name__}: {e}"}
    return json.dumps(result, ensure_ascii=False, default=str)

def ask_agent(question, max_steps=5, verbose=True):
    messages = [{"role": "system", "content": SYSTEM},
                {"role": "user", "content": question}]
    for _ in range(max_steps):
        resp = client.chat.completions.create(model=MODEL, messages=messages, tools=OAI_TOOLS)
        msg = resp.choices[0].message
        if not msg.tool_calls:
            return msg.content
        messages.append(msg.model_dump(exclude_none=True))
        for tc in msg.tool_calls:
            # if verbose:
                # print(f"🔧 {tc.function.name}({tc.function.arguments})")
            messages.append({"role": "tool", "tool_call_id": tc.id,
                             "content": run_tool(tc.function.name, tc.function.arguments)})
    return "Не удалось получить ответ за допустимое число шагов."

## 11. Агентті тексеру

Алдымен бір сұрақпен бастаңыз. Экранда алдымен `🔧 ...` жолы, содан кейін жауап шығуы керек.

In [55]:
print(ask_agent("990740002243 ұйымының бағасы қымбат сатып алуларын тап"))

Осы ұйым бойынша қымбат сатып алулар көп табылды: дәрілер мен медициналық бұйымдарда да, ерекше сипаттағы тауарларда да баға қалыпты деңгейден қатты жоғары. Ең айқындары — «Магния сульфат», бірнеше түрлі «Игла», «Наконечник», «Бинт» және кейбір жабдықтар.  

Жалпы 3326 қымбаттау жағдайы анықталды, оның ішінен ең күшті ауытқулар жоғарыда көрсетілді; кейбір позициялар бойынша дерек жеткілікті, ал сирек тауарлар санатында салыстыратын эталон жоқ.


Жұмыс істесе, баға мен көлем бойынша барлық сұрақты және қазақша сұраққа орысша жауапты тексеріңіз.

In [56]:
questions = [
    # Баға
    f"Какая справедливая цена по строке договора {lid_homo}?",
    f"Какая справедливая цена по строке договора {lid_srv}?",
    f"Справедливая цена по ЕНСТРУ {code_het}",
    f"Найди закупки с отклонением цены > 30% по ЕНСТРУ {code_homo}",
    "Найди аномалии цен на работы",
    "Покажи самые сильные завышения цен на услуги в 2025 году",
    f"{bin_top} ұйымының бағасы қымбат сатып алуларын тап",
    # Көлем
    f"Есть ли у организации {vol_bin} нетипичный рост объёмов закупок в {last_year} году?",
    f"Как менялось количество по ЕНСТРУ {vol_code} у организации {vol_bin}?",
    "Найди самые сильные завышения количества услуг по сравнению с прошлыми годами",
    f"{vol_bin} ұйымы қай тауарлардың санын негізсіз көбейтті?",
]
for q in questions:
    print("Вопрос:", q)
    print(ask_agent(q))
    print("─" * 80)

Вопрос: Какая справедливая цена по строке договора 53480920?
По этой строке фактическая цена 56 000 тенге. Справедливая цена по сравнению с другими договорами — медиана 51 249 тенге, средняя 81 605 тенге; цена в пределах нормы, явного завышения нет.
────────────────────────────────────────────────────────────────────────────────
Вопрос: Какая справедливая цена по строке договора 53608351?
По этой строке цена выглядит сильно выше обычного уровня: фактическая цена 13 888 400 тенге, а ориентир по другим закупкам — диапазон 704 605–3 821 096 тенге, медиана 2 006 629 тенге. Это примерно в 3,6 раза выше верхней границы диапазона, поэтому строка требует проверки.
────────────────────────────────────────────────────────────────────────────────
Вопрос: Справедливая цена по ЕНСТРУ 205952.100.000177
По коду ЕНСТРУ 205952.100.000177 справедливая цена зависит от единицы измерения и даётся диапазоном: для 704 — 66 651–334 311 тг, для 778 — 86 818–226 290 тг, для 796 — 541 669–900 130 тг, для 872 — 5